In [ ]:
import json
import sys
import os
from IPython.display import Image 
from IPython.display import clear_output, display

experiment = "cub200_experiment_llama_wo_human"
all_images_path = "data/cub200/images/"
labeled_images_file = f'checkpoints/{experiment}/classification/cub200/labeled_images.txt'

with open(f'checkpoints/{experiment}/classification/cub200/test_samples_old.json', 'r') as f:
    test_samples = json.load(f)

with open(f'checkpoints/{experiment}/classification/cub200/test_set200.txt', 'r') as f:
    image_files_names = f.read().splitlines()

labeled_images = set()
if os.path.exists(labeled_images_file):
    with open(labeled_images_file, 'r') as f:
        labeled_images = set(line.strip() for line in f if line.strip())

remaining_images = [img for img in image_files_names if img not in labeled_images]
scores = json.load(open(f'checkpoints/{experiment}/classification/cub200/test_samples_evaluation.json', 'r'))
total_remaining = len(remaining_images)

if total_remaining == 0:
    print("✓ All images have been labeled!")
    sys.exit()

print(f"📊 Total images to label: {total_remaining}")
print("-" * 50)
print("\n-----------------Instructions-----------------")
print("Binary concepts for all attributes not related to markings.")
print('''1/0 for all attributes with no markings
       olive brown = gray;  orange = reddish
        with markings:
        -	Color correct: 0.7
        -	Marking correct: 0.3
        - Both correct: 1
        ''')
print("----------------------------------------------\n")

📊 Total images to label: 18
--------------------------------------------------

-----------------Instructions-----------------
Binary concepts for all attributes not related to markings.
1/0 for all attributes with no markings
       olive brown = gray;  orange = reddish
        with markings:
        -	Color correct: 0.7
        -	Marking correct: 0.3
        - Both correct: 1
        
----------------------------------------------



In [47]:
reference_scores = json.load(open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_samples_evaluation_all.json', 'r'))
reference_concepts = json.load(open('checkpoints/cub200_experiment_llama_core/classification/cub200/test_samples_old.json', 'r'))

In [ ]:
for idx, image_file in enumerate(remaining_images):
    image_folder_id = image_file.split('/')[0]
    image_file_name = image_file.split('/')[1]
    image_path = all_images_path + image_folder_id + '/' + image_file_name

    attributes = test_samples[image_file]
    image = Image(filename=image_path, width=300, height=300)
    
    for key in attributes:
        clear_output(wait=True)
        
        # Display image and current instructions
        display(image)
        print(f"\n📌 Image: {image_file}")
        print(f"🔢 Progress: {idx+1}/{total_remaining}")
        
        # Get reference score if available
        ref_score = reference_scores.get(image_file, {}).get(key, "N/A")
        ref_concept = reference_concepts.get(image_file, {}).get(key, "N/A")
        current_value = attributes[key]
        
        # Format table
        print("\n" + "=" * 80)
        print(f"{'REFERENCE':<40} | {'CURRENT':<40}")
        print("-" * 80)
        print(f"{'Attribute: ' + key:<40} | {'Attribute: ' + key:<40}")
        print("-" * 80)
        print(f"{'Reference Value: ' + str(ref_concept):<40} | {'Current Value: ' + str(current_value):<40}")
        print(f"{'Reference Score: ' + str(ref_score):<40} | {'Current Score: [Input]':<40}")
        print("=" * 80)

        if ref_concept == current_value:
            scores[image_file][key] = ref_score
            continue
        
        while True:
            try:
                user_input_str = input("\nEnter label (1=correct, 0=incorrect, or decimal for partial): ")
                if user_input_str == '':
                    print("Input cannot be empty. Please try again.")
                    continue
                user_input = float(user_input_str)
                break
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        if user_input == -1:
            sys.exit("Evaluation stopped by user.")
        
        scores[image_file][key] = user_input

        # Save after all attributes for this image are labeled
    with open(f'checkpoints/{experiment}/classification/cub200/test_samples_evaluation.json', 'w') as f:
        json.dump(scores, f, indent=4)
    
    # Add to labeled images file
    with open(labeled_images_file, 'a') as f:
        f.write(image_file + '\n')
    
    remaining_count = total_remaining - idx - 1
    
    clear_output(wait=True)
    print(f"✓ Image {idx+1} complete!")
    print(f"📍 Labeled: {image_file}")
    print(f"⏳ Remaining images: {remaining_count}")
    
    if remaining_count > 0:
        print("\nPress Enter to continue to the next image...")
        input()
    else:
        print("\n✓ Evaluation complete!")

print("\n✓ All images have been labeled!")


✓ Image 18 complete!
📍 Labeled: 023.Brandt_Cormorant/Brandt_Cormorant_0033_22975.jpg
⏳ Remaining images: 0

✓ Evaluation complete!

✓ All images have been labeled!
